# DataProto (协议)：

在强化学习（RLHF）训练中，数据非常复杂：
- 既有需要在 GPU 上计算的 Tensor（如 input_ids），
- 又有需要保留的原始文本（如 prompt），
- 还有各种元数据（如 temperature）。

##### DataProto 的出现就是为了解决这些异构数据的统一管理、切分和传输问题。

- 这是 VERL 框架定义的统一数据协议（Data Protocol）。
- 它是一个专门设计的类（Class），不仅仅是存数据，还定义了数据如何在 Actor、Critic、Reward Model 等不同模块之间传递。
- 结构：DataProto 内部通常包含三个主要部分：
| 属性名 | 数据类型 | 用途 | 示例 |
| :--- | :--- | :--- | :--- |
| `batch` | `TensorDict` | 计算核心。存放所有参与模型前向/反向传播的 Tensor 数据。支持像单个 Tensor 一样进行切片、拼接、设备转移（CPU/GPU）。 | `input_ids`, `attention_mask`, `values`, `log_probs` |
| `non_tensor_batch` | `Dict` | 辅助信息。存放无法放入 Tensor 或不需要计算的数据，通常是字符串或列表。 | 原始 `prompt` 文本, 图片路径, 数据库查询结果 |
| `meta_info` | `Dict` | 全局配置。存放与具体样本无关的全局信息或统计量。 | `temperature`, `n_samples`, `global_steps` |


###### 局限性与演进
<font color='red'>虽然 DataProto 在标准的 RLHF（如 PPO）中非常高效，但在Agent 训练场景下（特别是异步、长尾延迟场景），它的“按 Batch 组织数据”的方式可能会成为瓶颈。</font>



## 创建

In [ ]:
# 从字典转换
data = DataProto.from_single_ditc({'input_ids':tensor,'prompts':list_text})

## 切片和切分

In [ ]:
# 取前32个样本，tensor和列表会自动对齐
sub_data = data.slice(0,32)

## 数据清洗 Pop
-  取出 "prompts" 并把它从 data 中移除，用于后续处理

In [ ]:
# 取出 "prompts" 并把它从 data 中移除，用于后续处理
prompts = data.pop('prompts')

## 设备转移

In [ ]:
# 一键上 GPU
data = data.to("cuda")

# DataProto.from_single_dict(batch_dict)

from_single_dict (方法)：

- 这是一个静态工厂方法。它的作用是将那个“朴素”的 batch_dict 解析，并自动归类填充到 DataProto 的内部结构中。
- 目的：让后续的所有计算（如 actor_rollout_wg.generate_sequences）都能接收统一格式的数据，而不需要关心数据最初长什么样。